# Stage 2. Similarity matrices and leave-one-out validation

This is the core result. Four retrieval spaces are compared under an identical
probability-transfer step, so any difference in skill is attributable to the
retrieval space alone.

No Earth Engine session is needed.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

from fireanalogs.config import Config

cfg = Config.load(Path.cwd().parent / "config" / "default.yaml")
DATA = Path.cwd().parent / cfg["paths"]["data_root"]
RESULTS = Path.cwd().parent / cfg["paths"]["results_root"]
RESULTS.mkdir(parents=True, exist_ok=True)
print("data:", DATA)

In [ ]:
import pandas as pd
from fireanalogs import similarity
from fireanalogs.library import embedding_matrix, has_usable_coordinates

library = pd.read_csv(RESULTS / "fire_library_merged.csv", parse_dates=["event_date"])

cosine = similarity.cosine_similarity_matrix(embedding_matrix(library, cfg.n_emb_dims))
doy = similarity.circular_doy_distance(library["doy"].values)

if has_usable_coordinates(library):
    geo = similarity.haversine_km(library["lat"].values, library["lon"].values)
    geo_seasonal = similarity.geo_seasonal_distance(
        geo, doy, cfg.geo_norm_km, cfg.doy_norm_days
    )
else:
    geo = geo_seasonal = None
    print("Too few parsed centroids for the geographic baselines.")

print("cosine similarity range:", round(cosine.min(), 3), "to", round(cosine.max(), 3))

## How often does the prediction-time rule fall back?

Targets early in the season have fewer than K prior fires available, so their
candidate pool widens to all other fires. The manuscript reports these counts.

In [ ]:
from fireanalogs.retrieval import fallback_report

eligible = library.index[library["has_fuelslope"] & library["obs_high_frac"].notna()].to_numpy()
report = fallback_report(library["event_date"].values, eligible, cfg.k_default)
print(report)

In [ ]:
from fireanalogs import validation

loo = validation.leave_one_out(
    library, cosine, doy, geo, geo_seasonal,
    k=cfg.k_default, prior_only=cfg.prior_only,
)
loo.to_csv(RESULTS / "loo_results.csv", index=False)

metrics = validation.event_metrics(loo)
metrics.to_csv(RESULTS / "loo_metrics.csv")

print(f"fires evaluated : {len(loo)}")
print(f"mean pool size  : {loo['n_pool'].mean():.0f}")
metrics.round(4)

## Is embedding similarity itself informative about error?

A weak but significant rank association is what the manuscript reports. It
supports treating similarity as one signal among several rather than as a
guarantee of accuracy.

In [ ]:
relationship = validation.similarity_error_relationship(loo)
print(f"Pearson  r = {relationship['pearson_r']:.3f}  p = {relationship['pearson_p']:.4f}")
print(f"Spearman rho = {relationship['spearman_rho']:.3f}  p = {relationship['spearman_p']:.4f}")
relationship["by_quartile"].round(3)